# 一階刷題：角色立繪

依 `plan.md`「美術與內容」的流程：**比稿 → 定稿基本立繪 → 表情差分 → 去背輸出**。

- 執行前：「執行階段 → 變更執行階段類型」選 GPU（有 L4 或 A100 就選，T4 也能跑但比較慢）。
- 全部用 `diffusers` 在 notebook 裡直接生成，不開 WebUI，符合 Colab 的使用規定。
- 每張圖的模型、seed、prompt 都存進 Google Drive 的 `nmle-art/`，之後可以重現。
- 輸出的檔案放進 repo 的 `data/characters/<角色 id>/<表情>.webp`，網站就會自動換上。

由上往下執行。需要你決定的地方都標了 **✏️**。

第 1 格裝完套件會自動重啟工作階段（畫面顯示工作階段當機是正常的），接著從第 2 格往下執行。

如果第 3 格出現 `cannot import name ...` 之類的錯誤：選「執行階段 → 重新啟動工作階段」，再從第 2 格重跑。

In [ ]:
# 1. 安裝套件
# 新的執行階段第一次跑時會安裝套件，裝完自動重啟工作階段（畫面會顯示工作階段當機，這是正常的），
# 接著從第 2 格繼續。同一個執行階段裡再跑這格會直接跳過。
# 要重啟是因為 Colab 開機時已經載入舊版套件（例如 Pillow），裝新版後不重啟會新舊混用而載入失敗。
import os, importlib.util
# 除了看記號，也實際檢查去背套件在不在：記號還在、套件卻不見時（例如換了執行階段），步驟 4 會找不到 rembg
if not os.path.exists('/content/.nmle_ready') or importlib.util.find_spec('rembg') is None:
    !pip install -q -U diffusers transformers accelerate safetensors "rembg[cpu]"
    # Colab 內建的 torchao 比新版 diffusers 需要的舊，裝著會讓 diffusers 載入失敗；我們用不到它（模型量化用的）
    !pip uninstall -y -q torchao
    open('/content/.nmle_ready', 'w').close()
    print('安裝完成，正在重新啟動工作階段……請從第 2 格繼續執行')
    os.kill(os.getpid(), 9)
else:
    print('套件已經裝好，請執行第 2 格')

In [ ]:
# 2. GPU 與 Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 角色、模型、表情設定（角色 tag 來自 characters.md）
import gc, json, time
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display
from huggingface_hub import hf_hub_download
from diffusers import (StableDiffusionXLPipeline, AutoPipelineForInpainting,
                       EulerDiscreteScheduler, EulerAncestralDiscreteScheduler)

OUT = Path('/content/drive/MyDrive/nmle-art')
OUT.mkdir(parents=True, exist_ok=True)
WIDTH, HEIGHT = 832, 1216

# 角色外觀放在 prompt 前段：SDXL 的文字編碼器只讀前 75 個 token，超過的會被默默丟掉
CHARACTERS = {
    'yiqing': {  # 夏以晴
        'tags': 'orange hair, short hair, bob cut, hairclip, amber eyes, athletic female, orange track jacket, '
                'open jacket, white t-shirt, track pants, sneakers, wristband, smile',  # 拿掉 energetic：Animagine 會畫出滿版特效背景
        'face': 'orange hair, short hair, bob cut, hairclip, amber eyes',
    },
    'fei': {  # 顏緋
        # 只寫 scrubs 常被畫成長洋裝或藍色，明確寫上衣加長褲
        'tags': 'red hair, medium hair, side braid, red eyes, tsurime, red scrubs, scrub top, scrub pants, labcoat, '
                'open labcoat, pen behind ear, crossed arms, slender',
        'face': 'red hair, medium hair, side braid, red eyes, tsurime, pen behind ear',
    },
    'jingcheng': {  # 沈靜澄
        # 原本 89 token，刪掉 side part、semi-rimless eyewear、slender、sweater，navy blue 改 blue，剛好 75
        'tags': 'black hair, very long hair, straight hair, blue eyes, glasses, tall female, labcoat, open labcoat, '
                'blue turtleneck, black pants, holding book, expressionless',
        # 眼鏡一定在臉部遮罩內，每個表情都會重畫，鏡框要和基本立繪一致。
        # 定稿的 seed 123 是灰色細金屬全框：寫 semi-rimless eyewear 會被改畫成只有上框，
        # 只寫 glasses 會被畫成金框，鏡腳在遮罩外仍是灰色，顏色接不起來
        'face': 'black hair, long hair, straight hair, side part, blue eyes, glasses, grey-framed eyewear',
    },
    'zhixun': {  # 溫芷薰
        # 原本 83 token。刪掉 grey hair（和 light purple hair 衝突）、gentle smile（表情會重畫）、id card（有 lanyard 就夠）、
        # tall female 和 low，共 74。
        # 第一次只寫 open labcoat、針織衫寫 lavender cardigan：24 張全部沒有白袍，針織衫被畫成紫色長外套，
        # lavender 還畫出薰衣草花叢。所以 labcoat 要和 open labcoat 一起寫（同以晴、顏緋），針織衫改成穿在裡面的 sweater
        'tags': 'light purple hair, long hair, wavy hair, side ponytail, purple eyes, droopy eyes, mole under eye, '
                'labcoat, open labcoat, purple sweater, long skirt, lanyard',
        # 不寫 mole under eye：這個 tag 不分左右，重畫時淚痣會換邊或變成兩顆。
        # 淚痣改用步驟 3 的 KEEP_DARK 從基本立繪搬回來，每個表情的位置都一樣
        'face': 'light purple hair, long hair, wavy hair, purple eyes, droopy eyes',
        # 大姊姊類型容易被畫成刻意強調胸部，和輔導級的尺度規範衝突；只加在她身上，不影響其他角色的定稿參數
        'negative': 'large breasts, cleavage',
    },
    'ying': {  # 葉縈
        # 原本 76 token。刪掉 gentle smile（表情會重畫）、head tilt（身體姿勢）、breast pocket（容易讓構圖往胸部靠），
        # test tube 改成 holding test tube；補上白袍底下的 green sweater、long skirt，共 74。
        # 只寫過大的白袍、不寫裡面穿什麼，容易被畫成只穿一件白袍、露出腿
        'tags': 'dark green hair, long hair, low twintails, hair ribbon, green eyes, bags under eyes, '
                'oversized labcoat, sleeves past fingers, holding test tube, green sweater, long skirt',
        'face': 'dark green hair, long hair, green eyes, bags under eyes',
        'negative': 'bare legs, large breasts, cleavage',
    },
}
COMMON = '1girl, solo, mature female, full body, standing, looking at viewer, simple background, white background'

EXPRESSIONS = {
    'normal': 'closed mouth, light smile',
    'happy': 'smile, open mouth, happy',
    'shy': 'blush, looking away, embarrassed',
    'sad': 'frown, sad, looking down',
    'surprised': 'surprised, wide-eyed, open mouth',
}
# 角色專屬表情，同名的會蓋掉上面的共用表情。顏緋的 pout 原本含 crossed arms，那是身體動作，只重畫臉部時不需要
EXTRA_EXPRESSIONS = {
    'fei': {'pout': 'pout, blush, looking away'},
    # 酷系：平常面無表情，開心也只是淡淡一笑
    'jingcheng': {'normal': 'closed mouth, expressionless', 'happy': 'light smile, closed mouth'},
    # 病嬌的「眼神死」，用在吃醋台詞。原本的 head tilt 是頭部動作，只重畫臉部時不需要
    'ying': {'yandere': 'smile, shaded face, empty eyes'},
}

SAFETY_NEG = 'nsfw, nude, explicit, child, loli, aged down'
MODELS = {
    'illustrious': {  # 主力候選
        'repo': 'OnomaAIResearch/Illustrious-XL-v2.0', 'file': 'Illustrious-XL-v2.0.safetensors',
        'sampler': 'euler_a', 'vpred': False, 'cfg': 6.0, 'steps': 28,
        'prefix': 'masterpiece, best quality, absurdres, ', 'suffix': '',
        'negative': 'lowres, worst quality, bad quality, bad anatomy, bad hands, extra fingers, missing fingers, '
                    'deformed, jpeg artifacts, watermark, signature, text, logo, ' + SAFETY_NEG,
    },
    'noobai': {  # v-prediction：只能用 Euler，CFG 4–5（模型說明頁的建議）
        'repo': 'Laxhar/noobai-XL-Vpred-1.0', 'file': 'NoobAI-XL-Vpred-v1.0.safetensors',
        'sampler': 'euler', 'vpred': True, 'cfg': 4.5, 'steps': 30,
        'prefix': 'masterpiece, best quality, newest, absurdres, safe, ', 'suffix': '',
        'negative': 'worst quality, old, early, low quality, lowres, signature, username, logo, bad hands, '
                    'mutated hands, ' + SAFETY_NEG,
    },
    'animagine': {  # 比稿定案（10/15）。品質 tag 放最後（模型說明頁的格式）
        'repo': 'cagliostrolab/animagine-xl-4.0', 'file': 'animagine-xl-4.0-opt.safetensors',
        'sampler': 'euler_a', 'vpred': False, 'cfg': 5.0, 'steps': 28,
        'prefix': '', 'suffix': ', safe, masterpiece, high score, great score, absurdres',
        'negative': 'lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, fewer digits, '
                    'cropped, worst quality, low quality, low score, bad score, average score, signature, '
                    'watermark, username, blurry, scenery, motion lines, ' + SAFETY_NEG,  # 後兩個壓背景特效
    },
}

# 重跑這格時，先把上次載入的模型從 GPU 清掉：不然程式以為還沒載入、又載入一份，
# 新舊兩份 SDXL 同時佔著 GPU，T4（15 GB）會記憶體不足（OutOfMemoryError）
if 'free_gpu' in globals():
    free_gpu()
pipe, inpaint_pipe, current_model = None, None, None


def free_gpu():
    global pipe, inpaint_pipe, current_model
    pipe = inpaint_pipe = current_model = None
    gc.collect()
    torch.cuda.empty_cache()


def load_model(key):
    """載入模型（第一次會下載約 7 GB，要幾分鐘）。"""
    global pipe, inpaint_pipe, current_model
    if current_model == key:
        return
    free_gpu()
    m = MODELS[key]
    path = hf_hub_download(m['repo'], m['file'])
    pipe = StableDiffusionXLPipeline.from_single_file(path, torch_dtype=torch.float16)
    if m['vpred']:
        pipe.scheduler = EulerDiscreteScheduler.from_config(
            pipe.scheduler.config, prediction_type='v_prediction', rescale_betas_zero_snr=True)
    elif m['sampler'] == 'euler_a':
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    else:
        pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.to('cuda')
    inpaint_pipe = AutoPipelineForInpainting.from_pipe(pipe)  # 共用權重，不多佔記憶體
    current_model = key
    print('已載入', key)


def check_tokens(text, label):
    count = len(pipe.tokenizer(text).input_ids) - 2
    if count > 75:
        raise ValueError(f'{label} 有 {count} 個 token，超過 75，後面的 tag 會被丟掉。請刪減 tag。\n{text}')
    return count


def build_prompt(model_key, body):
    m = MODELS[model_key]
    prompt = m['prefix'] + body + m['suffix']
    check_tokens(prompt, 'prompt')
    return prompt


def negative_for(model_key, char_key):
    """模型的負向 prompt，加上角色專屬的負向 tag（沒有就不加）。"""
    extra = CHARACTERS[char_key].get('negative')
    negative = MODELS[model_key]['negative'] + (', ' + extra if extra else '')
    check_tokens(negative, 'negative prompt')
    return negative


def log(record):
    with open(OUT / 'log.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def generate(model_key, char_key, seed):
    """產生一張全身立繪，存到 Drive。同一組模型、角色、seed 會得到同一張圖。"""
    load_model(model_key)
    m = MODELS[model_key]
    prompt = build_prompt(model_key, CHARACTERS[char_key]['tags'] + ', ' + COMMON)
    negative = negative_for(model_key, char_key)
    image = pipe(prompt=prompt, negative_prompt=negative, width=WIDTH, height=HEIGHT,
                 guidance_scale=m['cfg'], num_inference_steps=m['steps'],
                 generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    path = OUT / 'base' / model_key / char_key / f'{seed}.png'
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path)
    log({'kind': 'base', 'model': model_key, 'char': char_key, 'seed': seed, 'prompt': prompt,
         'negative': negative, 'cfg': m['cfg'], 'steps': m['steps'], 'path': str(path), 'time': time.time()})
    return image


def contact_sheet(images, labels, cols=4, thumb_h=420):
    """縮圖排成一張，方便比較。"""
    thumbs = [im.resize((round(im.width * thumb_h / im.height), thumb_h)) for im in images]
    w = max(t.width for t in thumbs)
    rows = (len(thumbs) + cols - 1) // cols
    sheet = Image.new('RGB', (w * cols, (thumb_h + 28) * rows), 'white')
    draw = ImageDraw.Draw(sheet)
    for i, (t, label) in enumerate(zip(thumbs, labels)):
        x, y = (i % cols) * w, (i // cols) * (thumb_h + 28)
        sheet.paste(t, (x, y + 28))
        draw.text((x + 6, y + 6), str(label), fill='black')
    return sheet

print('設定完成')

## 步驟 1：比稿

三個模型 × 兩位角色 × 同一組 seed，憑眼睛挑一個模型。**之後五位角色都用同一個模型**，畫風才會統一。

看的重點：手和身體結構、臉好不好看、明顯是成年人、衣服有沒有照 tag、畫風喜不喜歡。

三個模型要各下載一次（每個約 7 GB），全部跑完大約 15–30 分鐘。

In [ ]:
SEEDS = [11, 22, 33, 44]  # ✏️ 想多比幾張就加 seed
for model_key in MODELS:
    for char_key in CHARACTERS:
        images = [generate(model_key, char_key, s) for s in SEEDS]
        print(f'=== {model_key} / {char_key}')
        display(contact_sheet(images, [f'{model_key} seed {s}' for s in SEEDS]))
free_gpu()

## 步驟 2：定稿基本立繪

用選定的模型，每位角色換很多 seed，挑一張當基本立繪。表情差分會從這張改臉部，身體、衣服、姿勢都沿用，所以要挑**全身完整、手正常、臉部清楚**的。

每位角色各跑一次：在下面那格改 `CHAR`，從這裡往下跑完步驟 2–4。定稿後把 seed、`FACE_BOX`、`STRENGTH`、`PICKS` 記進 repo 的 `characters.md`「定稿紀錄」。

Colab 中途斷線重開的話：先執行最上面的 1–3 格（第 1 格裝完會自動重啟，再從第 2 格繼續），再執行下面這格「選定模型與角色」，就能直接跳到你做到的步驟，已經生成的圖都存在 Google Drive，不用重跑。

In [ ]:
# 選定模型與角色
CHOSEN_MODEL = 'animagine'    # 比稿定案；'illustrious'、'noobai' 也還能用
CHAR = 'ying'                 # ✏️ 'yiqing'（夏以晴）、'fei'（顏緋）、'jingcheng'（沈靜澄）、'zhixun'（溫芷薰）、'ying'（葉縈）

In [ ]:
CANDIDATE_SEEDS = list(range(100, 124))  # ✏️ 24 張；都不滿意就換一段數字，例如 range(200, 224)

images = [generate(CHOSEN_MODEL, CHAR, s) for s in CANDIDATE_SEEDS]
for i in range(0, len(images), 8):
    display(contact_sheet(images[i:i + 8], [f'seed {s}' for s in CANDIDATE_SEEDS[i:i + 8]]))

## 步驟 3：表情差分

只重畫臉部，身體不動：把臉附近裁下來放大重畫，再用羽化遮罩貼回原圖。

先調 **FACE_BOX**：實際重畫的是框內的**藍色橢圓**，要包住眉毛到下巴。編髮、髮夾這類每張表情都要一樣的特徵留在橢圓外，否則每個表情都會重畫一次。

臉上有淚痣這類「位置固定的小點」，在 `KEEP_DARK` 填它的位置：重畫完後會把基本立繪上那一小塊比較深的顏色疊回去，每個表情的痣都在同一個位置，周圍的臉紅、膚色仍是新表情的（預覽圖上的綠色小圈）。不要把這類特徵寫進臉部 tag，模型會自己挑位置畫。

產生表情時先設 `ONLY = 'surprised'` 只跑一個表情：驚訝會把眼睛畫大，最容易超出橢圓。眼睛完整、邊緣沒有鬼影，再改回 `ONLY = None` 跑全部。

In [ ]:
BASE_SEED = None   # ✏️ 步驟 2 選定的 seed
FACE_BOX = None    # ✏️ (x, y, 寬, 高)，單位是像素，圖是 832×1216
KEEP_DARK = []     # ✏️ 臉部遮罩內要從基本立繪保留的深色小點（例如淚痣），[(x, y, 半徑)]；沒有就留空
# 每位角色都不同，不沿用上一位的值；用過的值記在 characters.md 的「定稿紀錄」
assert BASE_SEED is not None and FACE_BOX is not None, '先填 BASE_SEED 和 FACE_BOX'

base = Image.open(OUT / 'base' / CHOSEN_MODEL / CHAR / f'{BASE_SEED}.png').convert('RGB')
preview = base.copy()
x, y, w, h = FACE_BOX
ImageDraw.Draw(preview).rectangle((x, y, x + w, y + h), outline='red', width=4)
ImageDraw.Draw(preview).ellipse((x, y, x + w, y + h), outline='blue', width=3)  # 實際重畫的範圍
for kx, ky, kr in KEEP_DARK:  # 綠色小圈：要保留的淚痣，圈的中心要對準痣
    ImageDraw.Draw(preview).ellipse((kx - kr - 3, ky - kr - 3, kx + kr + 3, ky + kr + 3), outline='lime', width=1)
display(preview.resize((416, 608)))
display(preview.crop((max(0, x - 80), max(0, y - 80), x + w + 80, y + h + 80)))

In [ ]:
from PIL import ImageChops


def face_mask(size, box, feather):
    mask = Image.new('L', size, 0)
    x, y, w, h = box
    ImageDraw.Draw(mask).ellipse((x, y, x + w, y + h), fill=255)
    return mask.filter(ImageFilter.GaussianBlur(feather))


def inpaint_face(base, box, expression_tags, seed, strength=0.6, pad=96):
    load_model(CHOSEN_MODEL)
    m = MODELS[CHOSEN_MODEL]
    x, y, w, h = box
    x0, y0 = max(0, x - pad), max(0, y - pad)
    x1, y1 = min(base.width, x + w + pad), min(base.height, y + h + pad)
    crop = base.crop((x0, y0, x1, y1))
    scale = 1024 / max(crop.size)  # 臉在全身圖裡很小，放大到約 1024 重畫，細節才夠
    big = (round(crop.width * scale / 8) * 8, round(crop.height * scale / 8) * 8)
    local_box = (x - x0, y - y0, w, h)
    big_box = tuple(round(v * big[0] / crop.width) for v in local_box)

    body = f"1girl, solo, mature female, portrait, {CHARACTERS[CHAR]['face']}, {expression_tags}, looking at viewer"
    prompt = build_prompt(CHOSEN_MODEL, body)
    out = inpaint_pipe(prompt=prompt, negative_prompt=negative_for(CHOSEN_MODEL, CHAR),
                       image=crop.resize(big, Image.LANCZOS), mask_image=face_mask(big, big_box, 12),
                       width=big[0], height=big[1], strength=strength,
                       guidance_scale=m['cfg'], num_inference_steps=m['steps'],
                       generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    # 縮回原尺寸，只在羽化遮罩範圍內換掉，遮罩外完全是原圖
    patch = Image.composite(out.resize(crop.size, Image.LANCZOS), crop, face_mask(crop.size, local_box, 6))
    result = base.copy()
    result.paste(patch, (x0, y0))
    keep_dark(result, base)
    return result, prompt


def keep_dark(result, base):
    """把基本立繪上的淚痣疊回去：只在小圓內，每個像素取兩張圖裡比較深的顏色。
    痣比皮膚深，所以會留下來；周圍的皮膚、臉紅比原圖深或差不多，維持新表情的樣子。"""
    for kx, ky, kr in KEEP_DARK:
        pad = 2
        box = (kx - kr - pad, ky - kr - pad, kx + kr + pad + 1, ky + kr + pad + 1)
        mask = Image.new('L', (box[2] - box[0], box[3] - box[1]), 0)
        ImageDraw.Draw(mask).ellipse((pad, pad, pad + 2 * kr, pad + 2 * kr), fill=255)
        mask = mask.filter(ImageFilter.GaussianBlur(1))
        here = result.crop(box)
        result.paste(Image.composite(ImageChops.darker(here, base.crop(box)), here, mask), box[:2])


expressions = {**EXPRESSIONS, **EXTRA_EXPRESSIONS.get(CHAR, {})}
ONLY = 'surprised'        # ✏️ 先只跑一個表情確認 FACE_BOX，沒問題再改成 None 跑全部
if ONLY:
    expressions = {ONLY: expressions[ONLY]}
TRY_SEEDS = [1, 2, 3, 4]  # ✏️ 每個表情試幾張
STRENGTH = 0.6            # ✏️ 表情變化太小就調高（最多約 0.8），臉不像同一個人就調低
for name, tags in expressions.items():
    results = []
    for s in TRY_SEEDS:
        img, prompt = inpaint_face(base, FACE_BOX, tags, s, STRENGTH)
        path = OUT / 'expr' / CHOSEN_MODEL / CHAR / str(BASE_SEED) / name / f'{s}.png'
        path.parent.mkdir(parents=True, exist_ok=True)
        img.save(path)
        log({'kind': 'expression', 'model': CHOSEN_MODEL, 'char': CHAR, 'base_seed': BASE_SEED, 'expression': name,
             'seed': s, 'strength': STRENGTH, 'face_box': FACE_BOX, 'prompt': prompt, 'path': str(path)})
        x, y, w, h = FACE_BOX
        results.append(img.crop((max(0, x - 120), max(0, y - 120), x + w + 120, y + h + 160)))
    print('===', name)
    display(contact_sheet(results, [f'{name} seed {s}' for s in TRY_SEEDS], thumb_h=300))

## 步驟 4：去背並輸出

每個表情各挑一張。去背只算一次（用基本立繪），所有表情共用同一個透明遮罩，切換表情時輪廓才不會跳動。

- **兩個去背模型合併**：`isnet-anime` 邊緣平滑，但會把細髮絲之間的背景當成半透明的頭髮留下來（沈靜澄的長髮最明顯）；BiRefNet 認得出那是背景，但細髮絲的邊緣會變成鋸齒。所以以 `isnet-anime` 為主，BiRefNet 判斷為背景的地方挖掉。
- **去白邊**：去背後半透明的邊緣還混著原本的淺色背景，放在網站深色模式上會有一圈白邊。輸出前用背景色反推出真正的顏色（同 repo 的 `art/defringe.py`）。
- 最後的預覽放在深色背景上，白邊和沒去乾淨的背景比較看得出來。

輸出成 `<角色 id>/<表情>.webp` 並打包成 zip 下載。解壓後把整個 `<角色 id>` 資料夾放進 repo 的 `data/characters/`，例如 `data/characters/yiqing/happy.webp`。

In [ ]:
PICKS = {  # ✏️ 每個表情選的 seed（步驟 3 的圖）；None 代表直接用基本立繪當這個表情
    'normal': 1,
    'happy': 1,
    'shy': 1,
    'sad': 1,
    'surprised': 1,
    # 'pout': 1,     # 顏緋才有
    # 'yandere': 1,  # 葉縈才有
}
# 所有表情盡量用同一個 seed，臉型才一致；也要看遮罩邊緣有沒有多畫東西（例如耳環、鏡框變形）
missing = set(EXPRESSIONS) | set(EXTRA_EXPRESSIONS.get(CHAR, {}))
missing -= set(PICKS)
assert not missing, f'PICKS 少了這些表情：{missing}'

import shutil
import numpy as np
from scipy.ndimage import distance_transform_edt, gaussian_filter
from rembg import new_session, remove
from google.colab import files

# ---- 去白邊（和 repo 的 art/defringe.py 相同，改那邊要一起改這裡） ----
SOLID = 230  # 透明度高於這個值的像素顏色可信，不動


def smooth_fill(rgb, known, sigma):
    """用 known 像素的顏色平滑地填滿整張圖（高斯加權平均）；離太遠的退回最近像素的顏色。"""
    w = gaussian_filter(known.astype(float), sigma)
    filled = np.stack([gaussian_filter(rgb[..., c] * known, sigma) for c in range(3)], -1) / np.maximum(w, 1e-6)[..., None]
    far = w < 1e-3
    if far.any():
        _, (iy, ix) = distance_transform_edt(~known, return_indices=True)
        filled[far] = rgb[iy, ix][far]
    return filled


def defringe(image: Image.Image, solid: int = SOLID) -> Image.Image:
    rgba = np.array(image.convert('RGBA')).astype(float)
    rgb, alpha = rgba[..., :3], rgba[..., 3]
    clear = alpha == 0
    if not clear.any() or not (alpha >= solid).any():
        return image
    # 每個位置原本的背景色：全透明的地方還是去背前的背景，從那裡平滑地推過來（背景有漸層也能處理）
    bg = smooth_fill(rgb, clear, 6)
    # 半透明像素 = 前景 × a + 背景 × (1 - a)，用背景色反推前景（把混進來的背景白扣掉）
    a = (alpha / 255)[..., None]
    fg = np.clip((rgb - (1 - a) * bg) / np.maximum(a, 1e-3), 0, 255)
    # 很透明的像素反推不穩定，改用附近不透明部分的平滑顏色
    near = smooth_fill(rgb, alpha >= solid, 2)
    t = np.clip((a - 0.15) / 0.35, 0, 1)
    semi = alpha < solid
    out = rgb.copy()
    out[semi] = (t * fg + (1 - t) * near)[semi]
    rgba[..., :3] = out
    return Image.fromarray(rgba.round().astype(np.uint8), 'RGBA')


def combine_masks(isnet: Image.Image, birefnet: Image.Image, cut: float = 60, sigma: float = 1.5) -> Image.Image:
    """合併兩個去背模型的遮罩：以 isnet-anime 的平滑邊緣為主，BiRefNet 判斷為背景的地方挖掉。
    isnet-anime 會把髮絲之間的背景當成半透明的頭髮留下來；BiRefNet 能分辨那是背景，但細髮絲的邊緣會變成鋸齒。
    BiRefNet 的遮罩先稍微模糊，鋸齒才不會傳到結果上。"""
    b = gaussian_filter(np.asarray(birefnet.convert('L')).astype(float), sigma)
    keep = np.clip(b / cut, 0, 1)
    return Image.fromarray((np.asarray(isnet.convert('L')).astype(float) * keep).round().astype(np.uint8), 'L')


# 去背：isnet-anime 邊緣平滑，但會把髮絲間的背景留下來；BiRefNet 認得出那是背景，但細髮絲會變鋸齒。
# 兩個合併：以 isnet-anime 為主，BiRefNet 判斷為背景的地方挖掉。BiRefNet 失敗就只用 isnet-anime
alpha = remove(base, session=new_session('isnet-anime'), only_mask=True).convert('L')
try:
    alpha = combine_masks(alpha, remove(base, session=new_session('birefnet-general'), only_mask=True))
    print('去背：isnet-anime + birefnet-general')
except Exception as e:
    print('BiRefNet 失敗，只用 isnet-anime：', e)

export_dir = Path('/content/export') / CHAR
export_dir.mkdir(parents=True, exist_ok=True)
for name, seed in PICKS.items():
    src = base if seed is None else Image.open(
        OUT / 'expr' / CHOSEN_MODEL / CHAR / str(BASE_SEED) / name / f'{seed}.png').convert('RGB')
    rgba = src.copy()
    rgba.putalpha(alpha)
    rgba = defringe(rgba)  # 去白邊：要在還有原本背景色的時候做
    rgba.save(export_dir / f'{name}.webp', 'WEBP', quality=90, method=6)
    print(name, (export_dir / f'{name}.webp').stat().st_size // 1024, 'KB')

# 放在深色底上檢查去背邊緣（網站深色模式的背景色），白邊和沒去乾淨的背景比較看得出來
paths = sorted(export_dir.glob('*.webp'))
backdrop = Image.new('RGBA', base.size, (26, 25, 23, 255))
display(contact_sheet([Image.alpha_composite(backdrop, Image.open(p).convert('RGBA')).convert('RGB') for p in paths],
                      [p.stem for p in paths], cols=6, thumb_h=360))

zip_path = shutil.make_archive(f'/content/{CHAR}', 'zip', '/content/export', CHAR)
shutil.copy(zip_path, OUT)
files.download(zip_path)